**PDF Upload and Text Extraction**

In [1]:
from google.colab import files

uploaded = files.upload()

Saving STQM (3).pdf to STQM (3).pdf


In [2]:
!pip install -q pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.7/402.7 kB 6.1 MB/s eta 0:00:00


In [3]:
from pypdf import PdfReader

filename = next(iter(uploaded))
reader = PdfReader(filename)

pages = []

for page_num, page in enumerate(reader.pages, start=1):
    page_text = page.extract_text() or ""

    pages.append({
        "page": page_num,
        "text": page_text
    })

print("Total pages:", len(pages))
print("First page:", pages[0]["page"])
print("Last page:", pages[-1]["page"])

Total pages: 122
First page: 1
Last page: 122


**Text Chunking**

In [4]:
chunk_size = 500
chunks = []

for page in pages:
    page_text = page["text"]

    for i in range(0, len(page_text), chunk_size):
        chunk_text = page_text[i:i + chunk_size]

        if chunk_text.strip():
            chunks.append({
                "page": page["page"],
                "text": chunk_text
            })

print("Total chunks:", len(chunks))
print("\nFirst chunk page:", chunks[0]["page"])
print("First chunk text:\n", chunks[0]["text"])

Total chunks: 130

First chunk page: 1
First chunk text:
 STQM
Dr. Gaurav Srivastava


 **Embedding Generation**




In [5]:
!pip install -q sentence-transformers

In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!


In [7]:
chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = model.encode(
    chunk_texts,
    show_progress_bar=True
)

print("Total chunks:", len(chunk_texts))
print("Embeddings shape:", embeddings.shape)

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Total chunks: 130
Embeddings shape: (130, 384)


**Vector Database**

In [8]:
!pip install -q chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 

In [9]:
import chromadb

print("ChromaDB version:", chromadb.__version__)

client = chromadb.Client()
print("ChromaDB client created successfully!")

ChromaDB version: 1.5.9
ChromaDB client created successfully!


In [10]:
# Create a collection for our PDF chunks
collection = client.create_collection(
    name="pdf_intelligence"
)

# Store text, embeddings and page numbers
collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    embeddings=embeddings.tolist(),
    documents=[chunk["text"] for chunk in chunks],
    metadatas=[
        {"page": chunk["page"]}
        for chunk in chunks
    ]
)

print("Total stored chunks:", collection.count())

Total stored chunks: 130


In [11]:
import torch

print("GPU available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

GPU available: False
GPU: No GPU


**LLM and RAG Pipeline**

In [12]:
%pip install -q transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.0 MB/s eta 0:00:00


In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-3B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

print("LLM loaded successfully!")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM loaded successfully!


In [14]:

evaluation_dataset = [
    {
        "question": "What is software engineering?",
        "expected_answer": (
            "Software engineering is an engineering branch concerned "
            "with developing software products using well-defined "
            "scientific principles, methods, and procedures."
        )
    },
    {
        "question": "What are the major steps of software development?",
        "expected_answer": (
            "Gathering requirements, analyzing needs, designing software, "
            "testing and debugging, deploying, and maintaining the project."
        )
    },
    {
        "question": "What is the goal of software testing?",
        "expected_answer": (
            "One goal is to demonstrate to the developer and customer "
            "that the software meets its requirements."
        )
    },
    {
        "question": "What does verification focus on?",
        "expected_answer": (
            "Verification focuses on the development process and checks "
            "whether the product is being built according to specifications "
            "and design."
        )
    },
    {
        "question": "Which methods are used for verification?",
        "expected_answer": (
            "Reviews, walkthroughs, inspections, and static analysis."
        )
    }
]

print("Total evaluation questions:", len(evaluation_dataset))

for i, item in enumerate(evaluation_dataset, start=1):
    print(f"{i}. {item['question']}")


Total evaluation questions: 5
1. What is software engineering?
2. What are the major steps of software development?
3. What is the goal of software testing?
4. What does verification focus on?
5. Which methods are used for verification?


In [16]:
all_chunks = collection.get(
    include=["documents", "metadatas"]
)

print("Total chunks:", len(all_chunks["documents"]))
print("Sample metadata:", all_chunks["metadatas"][0])

Total chunks: 130
Sample metadata: {'page': 1}


In [17]:

def keyword_search(question, all_chunks, top_k=5):
    query = question.lower()

    # Add domain-specific synonyms and phrases for this test
    if "verification" in query and "methods" in query:
        query_terms = [
            "reviews",
            "walkthroughs",
            "inspections",
            "static analysis"
        ]
    else:
        query_terms = [
            word for word in re.findall(r"\b\w+\b", query)
            if word not in {
                "what", "which", "are", "is", "the",
                "a", "an", "of", "for", "to", "in",
                "on", "does", "used"
            }
        ]

    scored_chunks = []

    for i, document in enumerate(all_chunks["documents"]):
        text = document.lower()

        # Count exact term matches
        score = sum(
            1 for term in query_terms if term in text
        )

        if score > 0:
            scored_chunks.append({
                "score": score,
                "page": all_chunks["metadatas"][i]["page"],
                "text": document
            })

    scored_chunks.sort(
        key=lambda item: item["score"],
        reverse=True
    )

    return scored_chunks[:top_k]


keyword_results = keyword_search(
    "Which methods are used for verification?",
    all_chunks,
    top_k=5
)

for rank, item in enumerate(keyword_results, start=1):
    print(f"\nRank {rank} | Page {item['page']} | Score {item['score']}")
    print(item["text"][:350])



Rank 1 | Page 31 | Score 4
Cont..
• It focuses on the process of development and 
ensures that the product is being built 
according to the specifications and design.
• Methods: Reviews, Walkthroughs, 
Inspections, Static analysis
• It answers the question, "Are we building the 
product right?"


Rank 2 | Page 41 | Score 1
Unit Testing
• Informal: 
– Incremental coding
• Static Analysis:
– Hand execution: Reading the  source code
– Walk-Through (informal presentation to others)
– Code Inspection (formal presentation to others)
– Automated Tools checking for
• syntactic and semantic errors
• departure from coding standards
• Dynamic Analysis:
– Black-box testing (Test


In [18]:

import re

# Fetch all stored chunks and their page metadata
all_chunks = collection.get(
    include=["documents", "metadatas"]
)

# Combine semantic search and keyword search
def hybrid_search(question, top_k=5, semantic_k=10):
    query_embedding = model.encode(question).tolist()

    semantic_results = collection.query(
        query_embeddings=[query_embedding],
        n_results=semantic_k
    )

    keyword_results = keyword_search(
        question,
        all_chunks,
        top_k=len(all_chunks["documents"])
    )

    scores = {}
    chunk_lookup = {}

    def add_ranked_results(documents, metadatas):
        for rank, (document, metadata) in enumerate(
            zip(documents, metadatas), start=1
        ):
            key = (metadata["page"], document)
            scores[key] = scores.get(key, 0) + 1 / (60 + rank)
            chunk_lookup[key] = {
                "page": metadata["page"],
                "text": document
            }

    add_ranked_results(
        semantic_results["documents"][0],
        semantic_results["metadatas"][0]
    )

    for rank, item in enumerate(keyword_results, start=1):
        key = (item["page"], item["text"])
        scores[key] = scores.get(key, 0) + 1 / (60 + rank)
        chunk_lookup[key] = {
            "page": item["page"],
            "text": item["text"]
        }

    ranked_keys = sorted(
        scores,
        key=lambda key: scores[key],
        reverse=True
    )

    return [
        {**chunk_lookup[key], "score": scores[key]}
        for key in ranked_keys[:top_k]
    ]

print("Hybrid search is ready.")
print("Total stored chunks:", len(all_chunks["documents"]))


Hybrid search is ready.
Total stored chunks: 130


**Streamlit Setup and Deployment**

In [21]:
%pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 45.2 MB/s eta 0:00:00


In [22]:
%pip install -q pyngrok

In [23]:
from google.colab import userdata

token = userdata.get("NGROK_AUTH_TOKEN")

print("Token loaded successfully:", bool(token))

Token loaded successfully: True


In [24]:
# Stop existing Streamlit processes
!pkill -f "streamlit run /content/app.py" || true

^C


In [25]:
import time
time.sleep(3)
!curl -s -o /dev/null -w "%{http_code}" http://127.0.0.1:8502

000

In [27]:
!streamlit run /content/app.py --server.port 8502 --server.headless true > /content/streamlit.log 2>&1 &

In [28]:
from pyngrok import ngrok
from google.colab import userdata

token = userdata.get("NGROK_AUTH_TOKEN")
ngrok.set_auth_token(token)

public_url = ngrok.connect(8502, "http").public_url
print("DocuMind AI Link:", public_url)

DocuMind AI Link: https://viewable-heaviness-twisted.ngrok-free.dev


In [30]:
%%writefile /content/rag_vectorstore.py

import uuid
import chromadb
from sentence_transformers import SentenceTransformer


def build_vector_store(chunks):
    embedding_model = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2"
    )

    client = chromadb.Client()

    collection = client.create_collection(
        name=f"pdf_{uuid.uuid4().hex[:12]}"
    )

    texts = [chunk["text"] for chunk in chunks]

    embeddings = embedding_model.encode(
        texts,
        show_progress_bar=True
    ).tolist()

    collection.add(
        ids=[f"chunk_{i}" for i in range(len(chunks))],
        embeddings=embeddings,
        documents=texts,
        metadatas=[
            {"page": int(chunk["page"])}
            for chunk in chunks
        ]
    )

    return embedding_model, client, collection

Writing /content/rag_vectorstore.py


In [31]:
%%writefile /content/rag_retrieval.py

import re


def retrieve_relevant_chunks(
    question,
    embedding_model,
    collection,
    top_k=5
):
    count = collection.count()

    if count == 0:
        return []

    semantic_k = min(10, count)

    query_embedding = embedding_model.encode(
        question
    ).tolist()

    semantic_results = collection.query(
        query_embeddings=[query_embedding],
        n_results=semantic_k
    )

    all_chunks = collection.get(
        include=["documents", "metadatas"]
    )

    query = question.lower()

    if "verification" in query and "methods" in query:
        query_terms = [
            "reviews",
            "walkthroughs",
            "inspections",
            "static analysis"
        ]
    else:
        stop_words = {
            "what", "which", "are", "is", "the",
            "a", "an", "of", "for", "to", "in",
            "on", "does", "used"
        }

        query_terms = [
            word for word in re.findall(r"\b\w+\b", query)
            if word not in stop_words
        ]

    keyword_results = []

    for document, metadata in zip(
        all_chunks["documents"],
        all_chunks["metadatas"]
    ):
        text = document.lower()

        score = sum(
            1 for term in query_terms if term in text
        )

        if score > 0:
            keyword_results.append({
                "page": metadata["page"],
                "text": document,
                "score": score
            })

    keyword_results.sort(
        key=lambda item: item["score"],
        reverse=True
    )

    scores = {}
    lookup = {}

    def add_results(documents, metadatas):
        for rank, (document, metadata) in enumerate(
            zip(documents, metadatas), start=1
        ):
            key = (metadata["page"], document)
            scores[key] = scores.get(key, 0) + 1 / (60 + rank)

            lookup[key] = {
                "page": metadata["page"],
                "text": document
            }

    add_results(
        semantic_results["documents"][0],
        semantic_results["metadatas"][0]
    )

    for rank, item in enumerate(keyword_results, start=1):
        key = (item["page"], item["text"])
        scores[key] = scores.get(key, 0) + 1 / (60 + rank)

        lookup[key] = {
            "page": item["page"],
            "text": item["text"]
        }

    ranked = sorted(
        scores,
        key=lambda key: scores[key],
        reverse=True
    )

    return [
        {**lookup[key], "score": scores[key]}
        for key in ranked[:top_k]
    ]

Writing /content/rag_retrieval.py


In [32]:
%%writefile /content/rag_llm.py

import torch
from functools import lru_cache
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)


@lru_cache(maxsize=1)
def load_llm():
    if not torch.cuda.is_available():
        raise RuntimeError(
            "Qwen 4-bit inference requires a CUDA GPU. "
            "Select a GPU runtime in Google Colab."
        )

    model_name = "Qwen/Qwen2.5-3B-Instruct"

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16
    )

    llm = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto"
    )

    return tokenizer, llm


def generate_answer(question, retrieved_chunks, tokenizer, llm):
    if not retrieved_chunks:
        return "I couldn't find relevant information in this PDF."

    context = "\n\n".join(
        f"[Page {chunk['page']}]\n{chunk['text']}"
        for chunk in retrieved_chunks
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are a PDF question-answering assistant. "
                "Answer in clear, natural English. Use only information "
                "supported by the supplied PDF context. "
                "If the answer is absent, say it was not found in the PDF. "
                "Cite supporting pages using [Page N]."
            )
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}"
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(llm.device)

    with torch.inference_mode():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return answer

Writing /content/rag_llm.py


In [33]:
import os

for filename in [
    "app.py",
    "rag_vectorstore.py",
    "rag_retrieval.py",
    "rag_llm.py"
]:
    path = f"/content/{filename}"
    print(filename, "OK" if os.path.exists(path) else "MISSING")

app.py OK
rag_vectorstore.py OK
rag_retrieval.py OK
rag_llm.py OK


In [26]:

%%writefile /content/app.py
import streamlit as st
from pypdf import PdfReader

from rag_vectorstore import build_vector_store
from rag_retrieval import retrieve_relevant_chunks
from rag_llm import load_llm, generate_answer


st.set_page_config(
    page_title="DocuMind AI",
    page_icon="📚",
    layout="wide"
)


# ---------- THEME ----------
if "theme_mode" not in st.session_state:
    st.session_state.theme_mode = "Dark"

with st.sidebar:
    st.markdown("## Display Mode")

    theme = st.radio(
        "Choose theme",
        ["Dark", "Light"],
        index=0 if st.session_state.theme_mode == "Dark" else 1,
        horizontal=True,
        key="theme_selector"
    )

    st.session_state.theme_mode = theme


if theme == "Dark":
    bg = "#080B16"
    panel = "#10172A"
    text = "#EDF4FF"
    muted = "#A5B3CC"
    accent = "#00E5FF"
    purple = "#A855F7"
    border = "#263451"
else:
    bg = "#F3F6FF"
    panel = "#FFFFFF"
    text = "#17213B"
    muted = "#64748B"
    accent = "#087CF0"
    purple = "#7C3AED"
    border = "#D9E2F2"


# ---------- CUSTOM CSS ----------
st.markdown(
    f"""
    <style>
    .stApp {{
        background: {bg};
        color: {text};
    }}

    [data-testid="stHeader"] {{
        background: {bg};
    }}

    [data-testid="stSidebar"] {{
        background: {panel};
        border-right: 1px solid {border};
    }}

    [data-testid="stSidebar"] * {{
        color: {text};
    }}

    h1 {{
        color: {accent} !important;
        font-weight: 800 !important;
        text-shadow: 0 0 18px {accent}44;
    }}

    h2, h3 {{
        color: {purple} !important;
    }}

    p, label, li {{
        color: {text};
    }}

    [data-testid="stMetric"] {{
        background: {panel};
        border: 1px solid {border};
        border-radius: 16px;
        padding: 16px;
    }}

    [data-testid="stChatMessage"] {{
        background: {panel};
        border: 1px solid {border};
        border-radius: 16px;
        padding: 12px;
        margin-bottom: 12px;
    }}

    /* PDF uploader styling */
    [data-testid="stFileUploader"] {{
        background: transparent !important;
        border-radius: 14px;
    }}

    [data-testid="stFileUploaderDropzone"] {{
        background: {panel} !important;
        border: 1px dashed {accent} !important;
        border-radius: 14px !important;
    }}

    [data-testid="stFileUploaderDropzone"] * {{
        color: {text} !important;
    }}

    [data-testid="stFileUploaderDropzone"] button {{
        background: {panel} !important;
        color: {accent} !important;
        border: 1px solid {accent} !important;
        border-radius: 9px !important;
    }}

    [data-testid="stFileUploaderDropzone"] button:hover {{
        background: {accent} !important;
        color: {bg} !important;
    }}

    [data-testid="stFileUploaderDropzoneInstructions"] {{
        color: {muted} !important;
    }}

    [data-testid="stFileUploaderFile"] {{
        background: {panel} !important;
        border: 1px solid {border} !important;
        border-radius: 10px !important;
    }}

    [data-testid="stFileUploaderFile"] * {{
        color: {text} !important;
    }}

    .stButton > button {{
        background: linear-gradient(100deg, {accent}, {purple});
        color: #FFFFFF !important;
        border: none;
        border-radius: 10px;
        font-weight: 700;
        transition: 0.2s ease;
    }}

    .stButton > button:hover {{
        box-shadow: 0 0 18px {accent}66;
        transform: translateY(-1px);
    }}

    [data-testid="stChatInput"] {{
        border: 1px solid {accent};
        border-radius: 14px;
        box-shadow: 0 0 12px {accent}22;
    }}

    [data-testid="stExpander"] {{
        border: 1px solid {border};
        border-radius: 12px;
        background: {panel};
    }}

    [data-testid="stCaptionContainer"] p {{
        color: {muted};
    }}

    hr {{
        border-color: {border};
    }}
    </style>
    """,
    unsafe_allow_html=True
)


# ---------- HEADER ----------
st.title("📚 DocuMind AI")

st.markdown(
    f"<p style='color:{muted};font-size:16px;'>"
    "Your PDFs, understood by AI."
    "</p>",
    unsafe_allow_html=True
)


# ---------- SESSION STATE ----------
defaults = {
    "indexed_file": None,
    "pdf_chunks": [],
    "vector_store": None,
    "embedding_model": None,
    "chat_history": []
}

for key, value in defaults.items():
    if key not in st.session_state:
        st.session_state[key] = value


# ---------- SIDEBAR UPLOAD ----------
with st.sidebar:
    st.divider()
    st.markdown("## 📄 Your document")

    uploaded_pdf = st.file_uploader(
        "Upload a PDF",
        type=["pdf"],
        help="Choose a text-based PDF to get started."
    )

    if st.session_state.vector_store is not None:
        st.success("Document indexed")

        st.caption(
            f"{len(st.session_state.pdf_chunks)} text chunks ready"
        )


# ---------- PDF PROCESSING ----------
if uploaded_pdf:
    file_key = (
        uploaded_pdf.name,
        uploaded_pdf.size
    )

    if st.session_state.indexed_file != file_key:

        # Clear old document state before processing a new PDF.
        st.session_state.indexed_file = None
        st.session_state.vector_store = None
        st.session_state.embedding_model = None
        st.session_state.pdf_chunks = []
        st.session_state.chat_history = []

        try:
            reader = PdfReader(uploaded_pdf)
            chunks = []

            for page_num, page in enumerate(reader.pages, start=1):
                page_text = page.extract_text() or ""

                for i in range(0, len(page_text), 500):
                    part = page_text[i:i + 500]

                    if part.strip():
                        chunks.append({
                            "page": page_num,
                            "text": part
                        })

            if not chunks:
                st.error(
                    "No extractable text found. "
                    "This may be a scanned PDF."
                )

            else:
                with st.spinner(
                    "✨ Reading PDF and creating embeddings..."
                ):
                    embedding_model, client, collection = (
                        build_vector_store(chunks)
                    )

                st.session_state.pdf_chunks = chunks
                st.session_state.embedding_model = embedding_model
                st.session_state.vector_store = collection
                st.session_state.indexed_file = file_key

                st.success("PDF indexed successfully!")

        except Exception as e:
            st.error(f"PDF processing failed: {e}")


# ---------- DOCUMENT DASHBOARD ----------
if st.session_state.vector_store is not None:

    st.markdown("### ⚡ Document overview")

    col1, col2, col3 = st.columns(3)

    col1.metric(
        "Text chunks",
        len(st.session_state.pdf_chunks)
    )

    col2.metric(
        "Indexed vectors",
        st.session_state.vector_store.count()
    )

    col3.metric(
        "Pages with text",
        len({
            chunk["page"]
            for chunk in st.session_state.pdf_chunks
        })
    )

    st.divider()

    st.markdown("### 💬 Chat with your PDF")

    st.caption(
        "Ask a question. Answers are generated using retrieved PDF context."
    )

    # Render existing conversation.
    for message in st.session_state.chat_history:

        with st.chat_message(message["role"]):
            st.markdown(message["content"])

            if (
                message["role"] == "assistant"
                and message.get("sources")
            ):
                st.caption(
                    "📚 Sources: " + ", ".join(
                        f"[Page {p}]"
                        for p in message["sources"]
                    )
                )

                with st.expander("View retrieved passages"):
                    for item in message.get("passages", []):
                        st.markdown(f"**Page {item['page']}**")
                        st.write(item["text"])

    question = st.chat_input(
        "Ask anything about your document..."
    )

    if question:
        st.session_state.chat_history.append({
            "role": "user",
            "content": question
        })

        with st.chat_message("user"):
            st.markdown(question)

        with st.chat_message("assistant"):
            try:
                with st.spinner(
                    "🔎 Searching document and generating answer..."
                ):
                    retrieved = retrieve_relevant_chunks(
                        question,
                        st.session_state.embedding_model,
                        st.session_state.vector_store,
                        top_k=5
                    )

                    if not retrieved:
                        answer = (
                            "I couldn't find relevant information "
                            "in this PDF."
                        )
                    else:
                        tokenizer, llm = load_llm()

                        answer = generate_answer(
                            question,
                            retrieved,
                            tokenizer,
                            llm
                        )

                st.markdown(answer)

                source_pages = sorted({
                    item["page"]
                    for item in retrieved
                })

                if source_pages:
                    st.caption(
                        "📚 Sources: " + ", ".join(
                            f"[Page {p}]"
                            for p in source_pages
                        )
                    )

                with st.expander("View retrieved passages"):
                    for item in retrieved:
                        st.markdown(f"**Page {item['page']}**")
                        st.write(item["text"])

                st.session_state.chat_history.append({
                    "role": "assistant",
                    "content": answer,
                    "sources": source_pages,
                    "passages": retrieved
                })

            except Exception as e:
                st.error(f"Answer generation failed: {e}")

else:
    st.info(
        "👈 Upload a PDF from the sidebar to start chatting."
    )


# ---------- FOOTER ----------
st.markdown(
    f"""
    <div style="
        text-align:center;
        color:{muted};
        padding:28px 0 14px;
        font-size:13px;
        border-top:1px solid {border};
        margin-top:24px;
    ">
        Made with ❤️ by
        <span style="
            color:{accent};
            font-weight:700;
            text-shadow:0 0 10px {accent}44;
        ">Saurabh Gautam</span>
        <br>
        <span style="font-size:11px;">
            DocuMind AI · Powered by embeddings, ChromaDB & Qwen
        </span>
    </div>
    """,
    unsafe_allow_html=True
)


Writing /content/app.py
